For all questions 

- only use 2015 and 2016 data for limiting the amount of data to be crunched. Use `201[56]` for filtering.
- Duckdb auto reads date columns as string so note the casting
- Remember that you can use `CREATE OR REPLACE TABLE` whenever appropriate

In [45]:
%%duckdb

SELECT * EXCLUDE(tripduration,starttime,stoptime), 
strptime(starttime, ['%m/%d/%Y %H:%M','%m/%d/%Y %H:%M:%S','%Y-%m-%d %H:%M:%S']) as start_at,
strptime(stoptime, ['%m/%d/%Y %H:%M','%m/%d/%Y %H:%M:%S','%Y-%m-%d %H:%M:%S']) as stop_at
from 's3://tt-bootcamp-shared/nyc-bike-trip/201[56]*.parquet' limit 3

,start station id,start station name,start station latitude,start station longitude,end station id,end station name,end station latitude,end station longitude,bikeid,usertype,birth year,gender,start_at,stop_at
0,455,1 Ave & E 44 St,40.750020,-73.969053,265,Stanton St & Chrystie St,40.722293,-73.991475,18660,Subscriber,1960.0,2,2015-01-01 00:01:00,2015-01-01 00:24:00
1,434,9 Ave & W 18 St,40.743174,-74.003664,482,W 15 St & 7 Ave,40.739355,-73.999318,16085,Subscriber,1963.0,1,2015-01-01 00:02:00,2015-01-01 00:08:00
2,491,E 24 St & Park Ave S,40.740964,-73.986022,505,6 Ave & W 33 St,40.749013,-73.988484,20845,Subscriber,1974.0,1,2015-01-01 00:04:00,2015-01-01 00:10:00


In [10]:
%%duckdb

select count(1) from 's3://tt-bootcamp-shared/nyc-bike-trip/201[56]*.parquet' limit 10;

,count(1)
0,23783624


### Q1. Profiling

- [ ] Profile data by generating summary statistics on all columns.
   - Min/Max/Mean/q[25,50,75]
   - Null %
- [ ] Note any wierd data profile/distribution

### Q2. Cleaning & Trip-Duration Derivation

- [ ] Calculate trip duration using relevant columns
- [ ] Plot a histogram for a sample of data
- [ ] Eleminate outlinears by using a heuristics you like

### Q3. Temporal Analysis

- [ ] Build a day of week and hour matrix to see when people uses trips

### Q4. Speed Analysis

- [ ] Turn coordinates into distance and speed, and catch GPS/data errors.

### Q5. Net Flow Analysis to Decide on Rebalancing

- [ ] Turn coordinates into distance and speed, and catch GPS/data errors.

### Q6. Weather Data Enrichment

- [ ] Enrich daily trips with weather data
- [ ] Can you build a regressor between num trips and weather ?

In [114]:
%%duckdb


CREATE OR REPLACE TABLE weather AS
WITH raw AS (
    SELECT daily
    FROM read_json_auto(
        'https://archive-api.open-meteo.com/v1/archive?latitude=40.7128&longitude=-74.0060&start_date=2015-01-01&end_date=2016-12-31&daily=temperature_2m_max,temperature_2m_min,temperature_2m_mean,precipitation_sum&timezone=America/New_York'
    )
)
SELECT
    unnest(daily.time)::DATE          AS date,
    unnest(daily.temperature_2m_max)  AS temp_max_c,
    unnest(daily.temperature_2m_min)  AS temp_min_c,
    unnest(daily.temperature_2m_mean) AS temp_mean_c,
    unnest(daily.precipitation_sum)   AS precip_mm
FROM raw;

select * from weather limit 10

,date,temp_max_c,temp_min_c,temp_mean_c,precip_mm
0,2015-01-01,2.4,-4.4,-1.5,0.0
1,2015-01-02,5.3,-2.4,0.9,0.0
2,2015-01-03,6.8,-3.3,0.9,16.5
3,2015-01-04,13.4,6.6,9.6,7.2
4,2015-01-05,7.1,-4.0,1.2,0.0
5,2015-01-06,-4.4,-6.7,-5.7,1.2
6,2015-01-07,-4.2,-12.5,-7.2,0.0
7,2015-01-08,-6.1,-13.4,-10.1,0.0
8,2015-01-09,0.5,-8.6,-3.9,0.9
9,2015-01-10,-5.2,-9.8,-7.6,0.0
